# 02 — EDA

Ноутбук исследует значения датасета: target, пропуски, дубликаты, числовые и категориальные признаки. Повторно используемая логика импортируется из `src/ml_project`; определения функций в notebook не дублируются.

Конфигурация файлов, key, target и описаний столбцов хранится централизованно в `src/ml_project/config.py`.

In [ ]:
from pathlib import Path
import sys

from IPython.display import display

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate
    for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (candidate / "README.md").exists() and (candidate / "src").exists()
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project import (
    DataCatalog,
    DatasetProfiler,
    MarkdownDocument,
    build_data_blocks,
    build_eda_blocks,
)
from ml_project.config import (
    DATASETS,
    FIELD_DESCRIPTIONS,
    INFERENCE_DATASET,
    KEY,
    RAW_DIR,
    TARGET,
    TRAIN_DATASET,
)

print(f"Корень проекта: {PROJECT_ROOT}")

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# В текущем окружении включите после проверки рендеринга matplotlib.
ENABLE_PLOTS = False

In [ ]:
catalog = DataCatalog(PROJECT_ROOT, RAW_DIR, DATASETS)
catalog.validate()
datasets = catalog.load_all()

profiles = {
    name: DatasetProfiler(
        frame,
        name=name,
        key=KEY,
        target=TARGET if name == TRAIN_DATASET else None,
    )
    for name, frame in datasets.items()
}
train_profile = profiles[TRAIN_DATASET]
test_profile = profiles.get(INFERENCE_DATASET)

print("Профили созданы:", ", ".join(profiles))

## 1. Первый взгляд

In [ ]:
for name, frame in datasets.items():
    print(name.upper())
    display(frame.head())

## 2. Target

In [ ]:
if TARGET is not None and TARGET in train_profile.df.columns:
    target_report = train_profile.target_report()
    display(target_report.style.format({"share": "{:.2%}"}))
else:
    target_report = None
    print("Target не настроен или отсутствует — заполните src/ml_project/config.py")

In [ ]:
if ENABLE_PLOTS and TARGET is not None and TARGET in train_profile.df.columns:
    train_profile.plot_target()
    plt.show()
else:
    print("График target пропущен: ENABLE_PLOTS=False или target не настроен")

## 3. Дубликаты и ключи

In [ ]:
duplicate_reports = {
    name: profile.duplicate_report()
    for name, profile in profiles.items()
}
for name, report in duplicate_reports.items():
    print(name.upper())
    display(report)

## 4. Пропуски

In [ ]:
missing_reports = {
    name: profile.missing_report()
    for name, profile in profiles.items()
}
for name, report in missing_reports.items():
    print(name.upper())
    display(report.style.format({"missing_share": "{:.2%}"}))

In [ ]:
missing_comparison = DatasetProfiler.compare_missing(*profiles.values())
display(missing_comparison.style.format(precision=2))

if ENABLE_PLOTS and not missing_comparison.empty:
    missing_comparison.set_index("field").plot(kind="bar", figsize=(10, 4))
    plt.title("Доля пропусков по датасетам")
    plt.ylabel("Доля")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## 5. Числовые признаки

In [ ]:
numeric_report = train_profile.numeric_report()
display(numeric_report)

In [ ]:
if ENABLE_PLOTS:
    train_profile.plot_numeric_histograms()
    plt.show()
else:
    print("Гистограммы пропущены: ENABLE_PLOTS=False")

## 6. Категориальные признаки

In [ ]:
categorical_report = train_profile.categorical_report(max_unique=30)
display(categorical_report)

## 7. Синхронизация с документом

Следующая ячейка обновляет только автоматические блоки Snapshot, Target и качества в `docs/02_eda.md`. Интерпретации, гипотезы и ручные выводы не перезаписываются.

In [ ]:
eda_blocks = build_eda_blocks(
    catalog,
    profiles,
    train_dataset=TRAIN_DATASET,
)
updated_blocks = MarkdownDocument(
    PROJECT_ROOT / "docs" / "02_eda.md"
).update_blocks(eda_blocks)

print("Обновлены блоки:", ", ".join(updated_blocks))

## 8. Что заполнить вручную

Автоматический профиль сообщает **что обнаружено**, но не объясняет **почему**. В `docs/02_eda.md` вручную зафиксируйте интерпретацию дисбаланса, причины пропусков, аномалии, leakage-кандидаты, выводы и проверяемые гипотезы.